# Measuring the Blazin' 5

Colin Cowherd names five NFL teams each Friday. This notebook shows whether those games' implied win probabilities moved differently from the rest of the slate between 9 AM and 9 PM. Every line is from DraftKings.

The calculation is `analysis/primary-comparison.py`. This notebook only loads that result and lays it out.

In [1]:
import runpy
from pathlib import Path

import pandas as pd

# The script owns the test. This notebook only presents what that file already computed.
# Repo-root sessions see analysis/. A session started inside analysis/ sees the file beside the notebook.
script = next(
    path
    for path in (
        Path("analysis/primary-comparison.py"),
        Path("primary-comparison.py"),
    )
    if path.exists()
)
# run_path executes the file without treating it as __main__, so it does not print the CLI tables.
result = runpy.run_path(str(script))
table = result["table"]
summary = result["summary"]
sample_size = result["sample_size"]
config = result["config"]


## How a game is scored

A picked game is scored from the endorsed team's own moneyline. Every other game is scored from the morning favorite's moneyline. A positive number means that side became more likely to win.

Both sides of each game have a price in the files, including when Cowherd picks the underdog. The shift uses that side's price directly.

In [2]:
# Copy first. Renaming columns here must not change the table the script computed.
show = table.copy()
show["group"] = show["group"].str.title()
# Control rows have no endorsed team. "none" reads more clearly than a blank cell.
show["endorsed"] = show["endorsed"].fillna("none")
show = show.rename(
    columns={
        "group": "Group",
        "endorsed": "Endorsed side",
        "tracked": "Tracked side",
        "favorite": "Morning favorite",
        "morning_pct": "Win chance, 9 AM",
        "evening_pct": "Win chance, 9 PM",
        "shift_pp": "Shift (percentage points)",
        "Game": "Game",
    }
)
show.style.hide(axis="index").format(
    {
        # Win chances as percentages; the shift keeps a sign so a drop reads as negative.
        "Win chance, 9 AM": "{:.1f}%",
        "Win chance, 9 PM": "{:.1f}%",
        "Shift (percentage points)": "{:+.2f}",
    }
)

Group,Endorsed side,Tracked side,Morning favorite,"Moneyline, 9 AM","Moneyline, 9 PM",Shift (percentage points),Game
Picked,Falcons,Falcons,49ers,124,124,+0.00,49ers @ Falcons
Picked,Bengals,Bengals,ravens,-102,-108,+1.43,Bengals @ Ravens
Picked,Browns,Browns,browns,-130,-135,+0.93,Browns @ Titans
Picked,Chiefs,Chiefs,seahawks,120,124,-0.81,chiefs @ seahawks
Picked,Lions,Lions,lions,-170,-180,+1.32,packers @ lions
Control,none,jets,jets,-225,-225,+0.00,Dolphins @ Jets
Control,none,steelers,steelers,-175,-162,-1.80,Steelers @ Saints (Paris)
Control,none,broncos,broncos,-230,-238,+0.72,broncos @ cardinals
Control,none,panthers,panthers,-162,-155,-1.05,bucs @ panthers
Control,none,vikings,vikings,-278,-298,+1.33,colts @ vikings


## Did the five picks move differently?

The test compares the average shift of the five picked games with the average shift of the other games. Shapiro-Wilk and Levene choose a pooled t test, a Welch t test, or Mann-Whitney. The interval is for the difference in average shifts.

In [3]:
# Display names only. The script's summary keys stay the reproducible record.
labels = {
    "picked_n": "Picked games",
    "control_n": "Other games",
    "picked_mean_pp": "Average shift, picked",
    "control_mean_pp": "Average shift, other games",
    "difference_pp": "Difference, picked minus other",
    "ci_low_pp": "95% interval, low",
    "ci_high_pp": "95% interval, high",
    "cohen_d": "Cohen's d",
    "shapiro_picked_p": "Shapiro-Wilk p, picked",
    "shapiro_control_p": "Shapiro-Wilk p, other games",
    "levene_p": "Levene p",
    "test": "Test",
    "stat": "Test statistic",
    "p": "p-value",
    "reject_null": "Difference beyond chance",
}
percent_point = {
    "picked_mean_pp",
    "control_mean_pp",
    "difference_pp",
    "ci_low_pp",
    "ci_high_pp",
}
values = []
for key, label in labels.items():
    value = summary[key]
    # Shifts are shown in percentage points. p-values stay probabilities, rounded for reading.
    if key in percent_point:
        value = f"{value:.2f} percentage points"
    elif key in {"cohen_d", "stat"}:
        value = f"{value:.3f}"
    elif key.endswith("_p") or key == "p":
        value = f"{value:.3f}"
    elif key == "reject_null":
        value = "Yes" if value else "No"
    elif key.endswith("_n"):
        value = f"{int(value)}"
    values.append({"Item": label, "Value": value})
pd.DataFrame(values).style.hide(axis="index")

Item,Value
Picked games,5
Other games,8
"Average shift, picked",0.57 percentage points
"Average shift, other games",-0.28 percentage points
"Difference, picked minus other",0.86 percentage points
"95% interval, low",-0.44 percentage points
"95% interval, high",2.16 percentage points
Cohen's d,0.827
"Shapiro-Wilk p, picked",0.369
"Shapiro-Wilk p, other games",0.638


In [4]:
# The sentence follows reject_null, so a later rerun cannot keep a stale written conclusion.
verdict = (
    "On this slate the picked games and the other games look alike."
    if not summary["reject_null"]
    else "The picked games shifted differently from the other games."
)
print(
    f"{verdict} The picked games differed from the others by "
    f"{summary['difference_pp']:.2f} percentage points "
    f"(95% interval {summary['ci_low_pp']:.2f} to {summary['ci_high_pp']:.2f}). "
    # The script stores a short test name. This map is only the label printed below.
    f"{ {'student t': 'Student t', 'welch t': 'Welch t', 'mann-whitney u': 'Mann-Whitney U'}[summary['test']] } "
    f"p = {summary['p']:.3f}, Cohen's d = {summary['cohen_d']:.2f}."
)

On this slate the picked games and the other games look alike. The picked games differed from the others by 0.86 percentage points (95% interval -0.44 to 2.16). Student t p = 0.175, Cohen's d = 0.83.


## How large a sample the design calls for

Equal groups, two-sided test. Alpha and power are the values set in the script.

In [5]:
# These counts are what the design would need, not how many games this Friday had.
sized = sample_size.rename(
    columns={
        "d": "Cohen's d",
        "n_per_group": "Games per group",
        "total": "Games in total",
    }
)
print(
    f"Alpha {config['alpha']}, power {config['power']}. "
    f"This slate has {int(summary['picked_n'] + summary['control_n'])} games."
)
sized.style.hide(axis="index").format(
    {
        "Cohen's d": "{:.1f}",
        "Games per group": "{:.0f}",
        "Games in total": "{:.0f}",
    }
)

Alpha 0.05, power 0.8. This slate has 13 games.


Cohen's d,Games per group,Games in total
0.5,64,128
0.8,26,52
